In [14]:
import os
import pandas as pd
import numpy as np

In [15]:
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

In [16]:
from understatapi import UnderstatClient

# 22/23から25/26シーズンまでの4年間の試合データ
seasons = ["2022", "2023", "2024", "2025"]
all_matches = []

with UnderstatClient() as client:
    for season in seasons:
        matches = client.team(team="Tottenham").get_match_data(season=season)
        df_season = pd.DataFrame(matches)
        df_season["season"] = season
        all_matches.append(df_season)

df_raw = pd.concat(all_matches, ignore_index=True)

# Understatから取得した生データをCSVとして保存（raw）
df_raw.to_csv("data/raw/spurs_raw_matches_2022_2025.csv", index=False)

In [17]:
# 加工用に生のデータをコピー
df = df_raw.copy()
df.shape

(152, 11)

In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 152 entries, 0 to 151
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        152 non-null    str   
 1   isResult  152 non-null    bool  
 2   side      152 non-null    str   
 3   h         152 non-null    object
 4   a         152 non-null    object
 5   goals     152 non-null    object
 6   xG        152 non-null    object
 7   datetime  152 non-null    str   
 8   forecast  152 non-null    object
 9   result    152 non-null    str   
 10  season    152 non-null    str   
dtypes: bool(1), object(5), str(5)
memory usage: 12.2+ KB


In [19]:
# 辞書方のhとaからそれぞれチーム名を取得
df["home_team"] = df["h"].str["title"]
df["away_team"] = df["a"].str["title"]

# ホームとアウェイチームの得点数
df["home_goals"] = df["goals"].str["h"].astype(int)
df["away_goals"] = df["goals"].str["a"].astype(int)

# ホームとアウェイチームのxG（ゴール期待値）を取得
df["home_xG"] = df["xG"].str["h"].astype(float)
df["away_xG"] = df["xG"].str["a"].astype(float)

In [20]:
# スパーズのxGと被xGを算出
df["spurs_xG"] = np.where(df["home_team"] == "Tottenham", df["home_xG"], df["away_xG"])
df["spurs_xGA"] = np.where(df["home_team"] == "Tottenham", df["away_xG"], df["home_xG"])

In [21]:
# datetimeを文字列型から日付型に交換
df["datetime"] = pd.to_datetime(df["datetime"])

# 試合日とキックオフ時間を取得
df["match_date"] = df["datetime"].dt.date
df["kickoff_time"] = df["datetime"].dt.time

In [22]:
# クリーニング＆加工済みデータの保存（processed）
df_processed = df[["season", "match_date", "kickoff_time", "home_team", "away_team", "home_goals", "away_goals", "spurs_xG", "spurs_xGA", "result"]].copy()
df_processed.to_csv("data/processed/spurs_cleaned_match_data.csv", index=False)